# CWT-assisted Multiscale Stepwise Learning (MSL)

This notebook runs a small non-confidential synthetic example of the MSL workflow. It demonstrates CWT preprocessing, progressive Step 1 feature learning, low-frequency model fusion, differentiable seismic forward modelling, and evaluation. Replace the synthetic data loader with the field-data loader when using the private dataset.

In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

repo = Path.cwd()
if (repo / 'msl').exists():
    pass
elif (repo / 'msl_reproducibility' / 'msl').exists():
    repo = repo / 'msl_reproducibility'
elif (repo.parent / 'msl').exists():
    repo = repo.parent
sys.path.insert(0, str(repo))

from msl import (
    CWTConfig, ConvolutionalForwardModel, MSLNetwork, TrainConfig,
    build_demo_dataset, predict, regression_metrics, ricker_wavelet,
    split_bundle, train_model, cwt_channel_summary, architecture_rows,
)
torch.manual_seed(7)
print('Repository:', repo.resolve())

## 1. CWT configuration

The example uses the real Morlet wavelet (`morl`), a 1-ms sampling interval, and 80 uniformly spaced pseudo-frequency channels from 2 to 80 Hz.

In [ ]:
cwt_config = CWTConfig(
    wavelet='morl',
    sample_interval=0.001,
    f_min=2.0,
    f_max=80.0,
    n_channels=80,
)
summary = cwt_channel_summary(cwt_config)
print({k: summary[k] for k in ['wavelet', 'central_frequency_cycles_per_sample', 'sample_interval_s', 'frequency_range_hz', 'number_of_channels']})
print('First frequencies:', summary['frequencies_hz'][:4])
print('Last frequencies:', summary['frequencies_hz'][-4:])

In [ ]:
bundle = build_demo_dataset(n_traces=16, n_samples=128, cwt_config=cwt_config)
train_bundle, test_bundle = split_bundle(bundle, train_fraction=0.75)
print('Seismic:', tuple(bundle.seismic.shape))
print('CWT:', tuple(bundle.cwt.shape))
print('Low-frequency model:', tuple(bundle.low_frequency_model.shape))
print('Impedance:', tuple(bundle.impedance.shape))

## 2. Architecture inspection

Step 1 uses four progressively connected frequency branches. Step 2 concatenates the four branch outputs with the projected low-frequency model. All odd-kernel convolutions use symmetric padding so that the time dimension is preserved.

In [ ]:
for row in architecture_rows():
    print(row)

model = MSLNetwork()
print(model)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 3. Training with the forward constraint

The demonstration uses impedance MSE plus a weighted synthetic-seismic reconstruction loss. The forward model converts predicted impedance to reflection coefficients and convolves them with a fixed Ricker wavelet.

In [ ]:
forward_model = ConvolutionalForwardModel(
    ricker_wavelet(length=31, dominant_frequency=30.0, dt=0.001)
)
history = train_model(
    model,
    train_bundle,
    forward_model,
    TrainConfig(epochs=2, batch_size=4, learning_rate=5e-3, weight_decay=1e-6),
)
history

In [ ]:
prediction = predict(model, test_bundle)
metrics = regression_metrics(prediction, test_bundle.impedance)
metrics

In [ ]:
idx = 0
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
axes[0].plot(test_bundle.impedance[idx, 0].numpy(), label='target')
axes[0].plot(prediction[idx, 0].numpy(), label='prediction')
axes[0].set_title('Impedance')
axes[0].legend()
axes[1].imshow(test_bundle.cwt[idx].numpy(), aspect='auto', origin='lower')
axes[1].set_title('CWT channels')
axes[1].set_xlabel('Time sample')
axes[1].set_ylabel('Channel')
axes[2].plot(test_bundle.seismic[idx, 0].numpy())
axes[2].set_title('Observed seismic')
axes[2].set_xlabel('Time sample')
plt.show()

In [ ]:
# Optional: save weights after checking the experiment.
# output_dir = repo / 'outputs'
# output_dir.mkdir(parents=True, exist_ok=True)
# torch.save(model.state_dict(), output_dir / 'msl_demo_weights.pt')
print('Notebook execution completed. Uncomment the last three lines to save model weights.')